# EV STOCK PRICE COMPARISON (2021–2024)
##Porsche vs Tesla vs Lucid vs Rivian
# =========================================================
### Research Question:
###### Did Porsche's EV strategy shift reflect in its stock performance compared to Tesla, Lucid, and Rivian?



In [ ]:
# Import

import sys
!{sys.executable} -m pip install yfinance --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


I installed and imported all the libraries needed for this project. yfinance pulls real stock data directly from Yahoo Finance. Pandas handles data manipulation, and Plotly builds the interactive charts.

In [ ]:
# Pull Stock Data

TICKERS = ['P911.DE', 'TSLA', 'LCID', 'RIVN']
NAMES   = {
    'P911.DE': 'Porsche',
    'TSLA':    'Tesla',
    'LCID':    'Lucid',
    'RIVN':    'Rivian'
}
START = '2021-01-01'
END   = '2024-12-31'

raw    = yf.download(TICKERS, start=START, end=END, auto_adjust=True)['Close']
prices = raw.rename(columns=NAMES)
prices = prices.dropna(how='all')

print(f' Data pulled: {len(prices)} trading days')
print(f'Date range: {prices.index[0].date()} to {prices.index[-1].date()}')
print()
display(prices.head().round(2))

[*********************100%***********************]  4 of 4 completed


 Data pulled: 1020 trading days
Date range: 2021-01-04 to 2024-12-30



Ticker,Lucid,Porsche,Rivian,Tesla
Date,,,,
2021-01-04,100.4,NaN,NaN,243.26
2021-01-05,100.2,NaN,NaN,245.04
2021-01-06,100.1,NaN,NaN,251.99
2021-01-07,100.3,NaN,NaN,272.01
2021-01-08,100.3,NaN,NaN,293.34


In this cell here I used yfinance to pull 4 years of daily closing prices for all 4 stocks in one funcation call. This is called an API call instead of typing manually, the library fetches it automatically from Yahoo Finance. I selected closing price because it represents the final agreed market value of each trading day.

In [ ]:
# Annual Summary Table

rows = []
for year in [2021, 2022, 2023, 2024]:
    year_data = prices[prices.index.year == year]
    if year_data.empty:
        continue
    for stock in ['Porsche', 'Tesla', 'Lucid', 'Rivian']:
        if stock not in year_data.columns:
            continue
        col = year_data[stock].dropna()
        if col.empty:
            continue
        open_price    = round(col.iloc[0], 2)
        close_price   = round(col.iloc[-1], 2)
        annual_return = round((close_price - open_price) / open_price * 100, 1)
        rows.append({
            'Year':            year,
            'Stock':           stock,
            'Open (Jan 1)':    open_price,
            'Close (Dec 31)':  close_price,
            'Annual Return %': annual_return
        })

annual_summary = pd.DataFrame(rows)
annual_summary.to_csv('ev_stock_annual_summary.csv', index=False)

display_df = annual_summary.copy()
display_df['Annual Return %'] = display_df['Annual Return %'].map(lambda x: f'{x:+.1f}%')
print('Annual Stock Performance Summary')
display(display_df.style.set_properties(**{'text-align': 'center'}).hide(axis='index'))
print(' Exported to ev_stock_annual_summary.csv')

Annual Stock Performance Summary


Year,Stock,Open (Jan 1),Close (Dec 31),Annual Return %
2021,Tesla,243.260000,352.260000,+44.8%
2021,Lucid,100.400000,380.500000,+279.0%
2021,Rivian,100.730000,103.690000,+2.9%
2022,Porsche,75.250000,86.430000,+14.9%
2022,Tesla,399.930000,123.180000,-69.2%
2022,Lucid,409.300000,68.300000,-83.3%
2022,Rivian,102.720000,18.430000,-82.1%
2023,Porsche,87.060000,73.560000,-15.5%
2023,Tesla,108.100000,248.480000,+129.9%
2023,Lucid,61.700000,42.100000,-31.8%


 Exported to ev_stock_annual_summary.csv


This cell calculates each stock's opening price like the first trading day of the year and closing price which is the last trading day, then computes the annual return as a percentage. This tells us at a glance which company had the best and worst year. The table is also exported as CSV for the Excel workbook

In [ ]:
# Key Stats

stats_rows = []
for stock in ['Porsche', 'Tesla', 'Lucid', 'Rivian']:
    if stock not in prices.columns:
        continue
    col = prices[stock].dropna()
    start_price  = round(col.iloc[0], 2)
    end_price    = round(col.iloc[-1], 2)
    total_return = round((end_price - start_price) / start_price * 100, 1)
    stats_rows.append({
        'Stock':          stock,
        'Start Price':    start_price,
        'End Price':      end_price,
        'All-Time High':  round(col.max(), 2),
        'All-Time Low':   round(col.min(), 2),
        'Total Return %': f'{total_return:+.1f}%'
    })

stats_df = pd.DataFrame(stats_rows)
print('Full Period Key Stats (2021–2024)')
display(stats_df.style.set_properties(**{'text-align': 'center'}).hide(axis='index'))

Full Period Key Stats (2021–2024)


Stock,Start Price,End Price,All-Time High,All-Time Low,Total Return %
Porsche,75.250000,55.490000,109.780000,54.350000,-26.3%
Tesla,243.260000,417.410000,479.860000,108.100000,+71.6%
Lucid,100.400000,31.500000,580.500000,20.100000,-68.6%
Rivian,100.730000,13.580000,172.010000,8.400000,-86.5%


This summary shows each stock's performance across the full 4-year period where it started, where it ended, its highest and lowest points, and the total return. This gives a quick snapshot before diving into the detailed charts.

In [ ]:
# Chart 1: Raw Price Chart

colors = {
    'Porsche': '#3B82F6',
    'Tesla':   '#EF4444',
    'Lucid':   '#10B981',
    'Rivian':  '#F59E0B'
}

fig1 = go.Figure()
for stock in ['Porsche', 'Tesla', 'Lucid', 'Rivian']:
    if stock not in prices.columns:
        continue
    fig1.add_trace(go.Scatter(
        x=prices.index,
        y=prices[stock].round(2),
        name=stock,
        line=dict(color=colors[stock], width=2)
    ))

fig1.update_layout(
    title='EV Stock Prices (2021–2024) — Raw Closing Price',
    xaxis_title='Date',
    yaxis_title='Closing Price (local currency)',
    plot_bgcolor='white',
    hovermode='x unified',
    yaxis=dict(gridcolor='#F3F4F6')
)
fig1.show()

This chart shows the raw closing price of each stock over time. One important limitation is Porsche trades in euros on the Farkfurt exchange while the others trade in US dollars on NASDAQ, so the absolute price levels cannot be directly compared. That is why the next cell normalizes all stocks to the same starting point.

In [ ]:
# Chart 2: Normalized to 100

normalized = prices.copy()
for col in normalized.columns:
    first_valid = normalized[col].dropna().iloc[0]
    normalized[col] = (normalized[col] / first_valid) * 100

fig2 = go.Figure()
fig2.add_hline(y=100, line_dash='dot', line_color='gray',
               annotation_text='Starting point', annotation_position='left')

for stock in ['Porsche', 'Tesla', 'Lucid', 'Rivian']:
    if stock not in normalized.columns:
        continue
    fig2.add_trace(go.Scatter(
        x=normalized.index,
        y=normalized[stock].round(1),
        name=stock,
        line=dict(color=colors[stock], width=2.5)
    ))

fig2.update_layout(
    title='EV Stock Performance Normalized to 100 — Who grew the most?',
    xaxis_title='Date',
    yaxis_title='Indexed Value (start = 100)',
    plot_bgcolor='white',
    hovermode='x unified',
    yaxis=dict(gridcolor='#F3F4F6')
)
fig2.show()

print('Final indexed value:')
for stock in ['Porsche', 'Tesla', 'Lucid', 'Rivian']:
    if stock in normalized.columns:
        final  = normalized[stock].dropna().iloc[-1]
        change = final - 100
        print(f'  {stock:10}: {final:.1f}  ({change:+.1f}%)')

Final indexed value:
  Porsche   : 73.7  (-26.3%)
  Tesla     : 171.6  (+71.6%)
  Lucid     : 31.4  (-68.6%)
  Rivian    : 13.5  (-86.5%)


To fairly compare stocks with different prices and currencies, I rebased all four to a starting value of 100. The formula is: (price / first price) × 100. A value of 150 means the stock is up 50% from the start. A value of 60 means it's down 40%. This is standard practice in financial analysis when comparing assets across different price ranges.

In [ ]:
# Chart 3: Annual Returnns

normalized = prices.copy()
for col in normalized.columns:
    first_valid = normalized[col].dropna().iloc[0]
    normalized[col] = (normalized[col] / first_valid) * 100

fig2 = go.Figure()
fig2.add_hline(y=100, line_dash='dot', line_color='gray',
               annotation_text='Starting point', annotation_position='left')

for stock in ['Porsche', 'Tesla', 'Lucid', 'Rivian']:
    if stock not in normalized.columns:
        continue
    fig2.add_trace(go.Scatter(
        x=normalized.index,
        y=normalized[stock].round(1),
        name=stock,
        line=dict(color=colors[stock], width=2.5)
    ))

fig2.update_layout(
    title='EV Stock Performance Normalized to 100 — Who grew the most?',
    xaxis_title='Date',
    yaxis_title='Indexed Value (start = 100)',
    plot_bgcolor='white',
    hovermode='x unified',
    yaxis=dict(gridcolor='#F3F4F6')
)
fig2.show()

print('Final indexed value:')
for stock in ['Porsche', 'Tesla', 'Lucid', 'Rivian']:
    if stock in normalized.columns:
        final  = normalized[stock].dropna().iloc[-1]
        change = final - 100
        print(f'  {stock:10}: {final:.1f}  ({change:+.1f}%)')

Final indexed value:
  Porsche   : 73.7  (-26.3%)
  Tesla     : 171.6  (+71.6%)
  Lucid     : 31.4  (-68.6%)
  Rivian    : 13.5  (-86.5%)


This chart breaks down performance year by year for each stock. Rather than looking at the full period as one number, this reveals which years were strong and which were weak for each company and whether they moved together or independently.

In [ ]:
# Chart 4: 2022 Crash

crash = prices[(prices.index.year >= 2022) & (prices.index.year <= 2023)].copy()
for col in crash.columns:
    first = crash[col].dropna().iloc[0]
    crash[col] = (crash[col] / first) * 100

fig4 = go.Figure()
fig4.add_hline(y=100, line_dash='dot', line_color='gray')
fig4.add_vrect(
    x0='2022-01-01', x1='2022-12-31',
    fillcolor='red', opacity=0.05,
    annotation_text='2022 crash', annotation_position='top left'
)
for stock in ['Porsche', 'Tesla', 'Lucid', 'Rivian']:
    if stock not in crash.columns:
        continue
    fig4.add_trace(go.Scatter(
        x=crash.index,
        y=crash[stock].round(1),
        name=stock,
        line=dict(color=colors[stock], width=2.5)
    ))

fig4.update_layout(
    title='2022 Crash & 2023 Recovery — Who bounced back?',
    xaxis_title='Date',
    yaxis_title='Indexed Value (Jan 2022 = 100)',
    plot_bgcolor='white',
    hovermode='x unified',
    yaxis=dict(gridcolor='#F3F4F6')
)
fig4.show()

crash_2022 = prices[prices.index.year == 2022].copy()
print('Max drawdown in 2022:')
for stock in ['Porsche', 'Tesla', 'Lucid', 'Rivian']:
    if stock not in crash_2022.columns:
        continue
    col      = crash_2022[stock].dropna()
    start    = col.iloc[0]
    trough   = col.min()
    drawdown = (trough - start) / start * 100
    print(f'  {stock:10}: {drawdown:+.1f}%')


Max drawdown in 2022:
  Porsche   : -0.8%
  Tesla     : -72.7%
  Lucid     : -84.9%
  Rivian    : -82.7%


2022 was a difficult year for EV stocks due to rising interest rates, inflation, and supply chain issues. This cell zooms in on 2022-2023 and normalizes from January of 2022 to show how far each stock fell during the crash and how quickly it recovered. The max drawdown printed below the chart shows the worst point each stock reached during 2022.

In [ ]:
# Chart 5: Volatility

daily_returns = prices.pct_change()
volatility    = daily_returns.rolling(window=30).std() * 100

fig5 = go.Figure()
for stock in ['Porsche', 'Tesla', 'Lucid', 'Rivian']:
    if stock not in volatility.columns:
        continue
    fig5.add_trace(go.Scatter(
        x=volatility.index,
        y=volatility[stock].round(2),
        name=stock,
        line=dict(color=colors[stock], width=2)
    ))

fig5.update_layout(
    title='30-Day Rolling Volatility — Higher = Riskier',
    xaxis_title='Date',
    yaxis_title='Volatility (% daily std dev)',
    plot_bgcolor='white',
    hovermode='x unified',
    yaxis=dict(gridcolor='#F3F4F6')
)
fig5.show()

print('Average volatility over full period:')
for stock in ['Porsche', 'Tesla', 'Lucid', 'Rivian']:
    if stock in volatility.columns:
        avg = volatility[stock].mean()
        print(f'  {stock:10}: {avg:.2f}%')

/tmp/ipykernel_5062/4034538824.py:3: FutureWarning:

The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.



Average volatility over full period:
  Porsche   : 1.75%
  Tesla     : 3.60%
  Lucid     : 5.31%
  Rivian    : 4.84%


Volatility measures how much a stock's price jumps around day-to-day. I calculated a 30-day rolling standard deviation of daily returns and computed how spread out they are. A higher number means more unpredictable price swings, which equals higher risk. This helps answer which EV stock was the riskiest investment.

In [ ]:
# Chart 6: Correlation Heatmap

corr_matrix = daily_returns[['Porsche', 'Tesla', 'Lucid', 'Rivian']].corr().round(2)

fig6 = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns.tolist(),
    y=corr_matrix.index.tolist(),
    colorscale='RdBu',
    zmid=0, zmin=-1, zmax=1,
    text=corr_matrix.values,
    texttemplate='%{text}',
    textfont=dict(size=16, color='white')
))

fig6.update_layout(
    title='Correlation Matrix — Daily Returns (2021–2024)',
    width=500, height=450
)
fig6.show()

print('Correlation matrix:')
display(corr_matrix)

Correlation matrix:


Ticker,Porsche,Tesla,Lucid,Rivian
Ticker,,,,
Porsche,1.00,0.18,0.15,0.19
Tesla,0.18,1.00,0.38,0.47
Lucid,0.15,0.38,1.00,0.65
Rivian,0.19,0.47,0.65,1.00


The correlation matrix shows how closely two stocks move together daily. A score of 1.0 means they move in a perfect lockstep. A score of 0 means no relation. A score close to -1 indicates that they move in opposite directions. This is useful to understand whether Porsche moves with the broader EV market or independently.

In [ ]:
# Key Findings

findings_rows = []
for stock in ['Porsche', 'Tesla', 'Lucid', 'Rivian']:
    if stock not in prices.columns:
        continue
    col          = prices[stock].dropna()
    total_return = round((col.iloc[-1] - col.iloc[0]) / col.iloc[0] * 100, 1)
    avg_vol      = round(daily_returns[stock].std() * 100, 2)
    stock_annual = annual_summary[annual_summary['Stock'] == stock]
    best         = stock_annual.loc[stock_annual['Annual Return %'].idxmax()]
    worst        = stock_annual.loc[stock_annual['Annual Return %'].idxmin()]
    findings_rows.append({
        'Stock':          stock,
        'Total Return':   f'{total_return:+.1f}%',
        'Avg Daily Vol':  f'{avg_vol:.2f}%',
        'Best Year':      f"{int(best['Year'])} ({best['Annual Return %']:+.1f}%)",
        'Worst Year':     f"{int(worst['Year'])} ({worst['Annual Return %']:+.1f}%)"
    })

findings_df = pd.DataFrame(findings_rows)
print('Key Findings — Full Period (2021–2024)')
print('=' * 65)
display(findings_df.style.set_properties(**{'text-align': 'center'}).hide(axis='index'))

Key Findings — Full Period (2021–2024)


Stock,Total Return,Avg Daily Vol,Best Year,Worst Year
Porsche,-26.3%,1.82%,2022 (+14.9%),2024 (-24.7%)
Tesla,+71.6%,3.75%,2023 (+129.9%),2022 (-69.2%)
Lucid,-68.6%,5.84%,2021 (+279.0%),2022 (-83.3%)
Rivian,-86.5%,5.14%,2023 (+35.3%),2022 (-82.1%)


Finally this table pulls the key numbers from all the analysis above into one clean summary of total return, average daily volatility, best and worst year for each stock. This is the evidence base for the conclusion.